In [ ]:
# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

### Первый вариант кода:
- Долгая обработка файлов из-за циклов.
- Но если файлов в целом немного, то подойдёт

In [ ]:
import pandas as pd
import zipfile
import os
from pathlib import Path
from datetime import datetime

# Названия колонок для файлов СБИС (в выгрузке нет заголовков, задаём сами)
sbis_names = ["Дата",
            "Номер",
            "Сумма",
            "Статус",
            "Примечание",
            "Комментарий",
            "Контрагент",
            "ИНН/КПП",
            "Организация",
            "ИНН/КПП",
            "Тип документа",
            "Имя файла",
            "Дата",
            "Номер 1",
            "Сумма 1",
            "Сумма НДС",
            "Ответственный",
            "Подразделение",
            "Код",
            "Дата",
            "Время",
            "Тип пакета",
            "Идентификатор пакета",
            "Запущено в обработку",
            "Получено контрагентом",
            "Завершено",
            "Увеличение суммы",
            "НДC",
            "Уменьшение суммы",
            "НДС"
]

# Создаём папку для результатов, имя - сегодняшняя дата
result_path_for_folder = f'Результат/{datetime.now().strftime("%Y-%m-%d")}/'
Path(result_path_for_folder).mkdir(parents=True, exist_ok=True)

# Читаем все CSV из архива Входящие.zip (это выгрузка СБИС)
folder_zip_entrance = 'Входящие.zip'
archive_file_entrance = Path(os.getcwd()) / folder_zip_entrance

dfs_entrance = []
with zipfile.ZipFile(archive_file_entrance, 'r') as zip_ref:

    csv_files = [name for name in zip_ref.namelist() if name.endswith('.csv')]

    if not csv_files:
        raise ValueError('CSV-файлы в архиве не найдены')
    
    for item in csv_files:
        with zip_ref.open(item) as file:
            # skiprows=1 - первая строка в файлах СБИС пустая или служебная
            # header=None - названия колонок задаём из списка sbis_names
            df = pd.read_csv(file, encoding='windows-1251', sep=';', header=None, skiprows=1)
            dfs_entrance.append(df)

    if dfs_entrance:
        combined_df_sbis = pd.concat(dfs_entrance, ignore_index=True)
    else:
        print('Не удалось прочитать ни один файл')

# Присваиваем колонкам имена и меняем пробелы на подчёркивания
combined_df_sbis.columns = sbis_names
combined_df_sbis.columns = [i.replace(' ', '_') for i in combined_df_sbis.columns]

# ---------------------------------------------------------------------------------------
# Обработка файлов аптек

apteka_folder = "Аптеки/csv/correct/"
archive_file_apteki = Path(os.getcwd()) / folder_zip_apteki  # Путь к архиву с аптеками

with zipfile.ZipFile(archive_file_apteki, 'r') as zip_ref:
    # Ищем CSV только в папке Аптеки/csv/correct/ внутри архива
    csv_files = [item for item in zip_ref.namelist() if item.startswith(apteka_folder) and item.endswith('.csv')]

    if not csv_files:
        raise ValueError('Нет файлов csv')
    
    for file_path in csv_files:
        with zip_ref.open(file_path) as file_1:
            apteka = pd.read_csv(file_1, sep=';', encoding='windows-1251')

            # Добавляем пустые колонки, которые будем заполнять
            apteka["Номер счет-фактуры"] = ""
            apteka["Сумма счет-фактуры"] = ""
            apteka["Дата счет-фактуры"] = ""
            apteka["Сравнение дат"] = ""

            # Типы документов в СБИС, которые нам подходят
            docs = ["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]

            # Проходим по каждой строке аптеки
            for i, row in apteka.iterrows():
                number_nakl = row['Номер накладной']
                
                # Для поставщика ЕАПТЕКА добавляем /15 к номеру накладной
                if 'ЕАПТЕКА' in row['Поставщик']:
                    number_nakl += '/15'

                # Ищем совпадения в СБИС по номеру и типу документа
                sbis_filter = combined_df_sbis[
                    (combined_df_sbis['Номер'] == number_nakl) &
                    (combined_df_sbis['Тип_документа'].isin(docs))
                ]

                # Если ничего не нашли
                if sbis_filter.empty:
                    date_val = row['Дата накладной']
                    # Если дата накладной заполнена, а счёт-фактуры нет - это расхождение
                    if pd.notna(date_val) and str(date_val).strip() != '':
                        apteka.at[i, 'Сравнение дат'] = 'Не совпадает'
                    continue

                # Если нашли - берём первую запись
                invoice = sbis_filter.iloc[0]["Номер"]
                summ = sbis_filter.iloc[0]["Сумма"]
                
                # В колонке "Дата" лежит три даты, нам нужна вторая (индекс 1)
                date = sbis_filter.iloc[0]["Дата"].iloc[1]
                # Приводим к формату ДД.ММ.ГГГГ (в СБИС год из двух цифр)
                date = datetime.strptime(date, "%d.%m.%y").strftime("%d.%m.%Y")

                # Записываем найденные данные
                apteka.at[i, "Номер счет-фактуры"] = invoice
                apteka.at[i, "Сумма счет-фактуры"] = summ
                apteka.at[i, "Дата счет-фактуры"] = date
                
                # Сравниваем дату накладной с датой счёта-фактуры
                apteka.at[i, "Сравнение дат"] = "" if (date == apteka.at[i, 'Дата накладной']) else "Не совпадает!"

            # Колонки для итогового файла (порядок важен)
            apteka_columns = ['№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
        'Дата приходного документа', 'Номер приходного документа',
        'Дата накладной', 'Номер накладной', 'Номер счет-фактуры',
        'Сумма счет-фактуры', 'Кол-во',
        'Сумма в закупочных ценах без НДС', 'Ставка НДС поставщика',
        'Сумма НДС', 'Сумма в закупочных ценах с НДС', 'Дата счет-фактуры', 'Сравнение дат']
            
            # Оставляем только нужные колонки
            apteka = apteka[apteka_columns]
            
            # Сохраняем результат в Excel
            file_name = os.path.basename(file_path)
            base_name = os.path.splitext(file_name)[0]
            output_path = f"{result_path_for_folder}{base_name} - результат.xlsx"
            apteka.to_excel(output_path, index=False)
            print(f'{file_name} Обработан!')

### Первый вариант кода:
- Быстрая обработка данных за счёт векторизации и join.
- Подходит для большего количества файлов и приоритета в скорости обработки файлов

In [ ]:
import pandas as pd
import zipfile
import os
from pathlib import Path
from datetime import datetime

# Настройки pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Названия колонок для файлов СБИС
sbis_names = [
    "Дата", "Номер", "Сумма", "Статус", "Примечание", "Комментарий",
    "Контрагент", "ИНН/КПП", "Организация", "ИНН/КПП", "Тип документа",
    "Имя файла", "Дата", "Номер 1", "Сумма 1", "Сумма НДС",
    "Ответственный", "Подразделение", "Код", "Дата", "Время",
    "Тип пакета", "Идентификатор пакета", "Запущено в обработку",
    "Получено контрагентом", "Завершено", "Увеличение суммы",
    "НДC", "Уменьшение суммы", "НДС"
]

# Колонки для итогового Excel-файла
apteka_columns = [
    '№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
    'Дата приходного документа', 'Номер приходного документа',
    'Дата накладной', 'Номер накладной', 'Номер счет-фактуры',
    'Сумма счет-фактуры', 'Кол-во', 'Сумма в закупочных ценах без НДС',
    'Ставка НДС поставщика', 'Сумма НДС', 'Сумма в закупочных ценах с НДС',
    'Дата счет-фактуры', 'Сравнение дат'
]

# Типы документов в СБИС, которые нам нужны
docs = ["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]

# Создаём папку для результатов с сегодняшней датой
result_path_for_folder = f'Результат/{datetime.now().strftime("%Y-%m-%d")}/'
Path(result_path_for_folder).mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. Загрузка всех файлов СБИС из архива Входящие.zip

dfs_entrance = []
with zipfile.ZipFile('Входящие.zip', 'r') as zip_ref:
    for item in [n for n in zip_ref.namelist() if n.endswith('.csv')]:
        with zip_ref.open(item) as f:
            # skiprows=1 - первая строка служебная, пропускаем
            # header=None - названия колонок зададим сами
            df = pd.read_csv(f, encoding='windows-1251', sep=';', header=None, skiprows=1)
            dfs_entrance.append(df)

# Объединяем все файлы в один датафрейм
combined_df_sbis = pd.concat(dfs_entrance, ignore_index=True)
combined_df_sbis.columns = sbis_names
combined_df_sbis.columns = [i.replace(' ', '_') for i in combined_df_sbis.columns]

# ============================================================

# 2. Подготовка данных СБИС для быстрого поиска
# Делаем один раз, чтобы не фильтровать для каждой строки

# Оставляем только нужные типы документов
sbis_filtered = combined_df_sbis[combined_df_sbis['Тип_документа'].isin(docs)].copy()

# Группируем по номеру, берём первую запись (убираем дубликаты)
sbis_grouped = sbis_filtered.groupby('Номер').first().reset_index()

# Извлекаем вторую дату - это дата счёта-фактуры
# В колонке "Дата" лежит три даты, нам нужна вторая (индекс 1)
sbis_grouped['Дата_2'] = sbis_filtered.groupby('Номер')['Дата'].apply(
    lambda x: x.iloc[0].iloc[1] if len(x.iloc[0]) > 1 else ''
).values

# Приводим дату к формату ДД.ММ.ГГГГ (в СБИС ДД.ММ.ГГ, в аптеке ДД.ММ.ГГГГ)
sbis_grouped['Дата_формат'] = pd.to_datetime(
    sbis_grouped['Дата_2'], 
    format='%d.%m.%y', 
    errors='coerce'
).dt.strftime('%d.%m.%Y').fillna('')

# Создаём справочник: номер, сумма, дата
sbis_lookup = sbis_grouped[['Номер', 'Сумма', 'Дата_формат']].copy()
sbis_lookup.columns = ['Номер_поиска', 'Сумма_счета', 'Дата_счета']

# ============================================================
# 3. Обработка каждого файла аптеки

with zipfile.ZipFile('Аптеки.zip', 'r') as zip_ref:
    # Ищем CSV только в папке Аптеки/csv/correct/
    csv_files = [i for i in zip_ref.namelist() 
                 if i.startswith("Аптеки/csv/correct/") and i.endswith('.csv')]
    
    if not csv_files:
        print('Файлы не найдены в папке Аптеки/csv/correct/')
    
    for file_path in csv_files:
        with zip_ref.open(file_path) as f:
            apteka = pd.read_csv(f, sep=';', encoding='windows-1251')
            
            # Создаём ключ для поиска в СБИС
            # Для поставщика ЕАПТЕКА добавляем /15 к номеру накладной
            apteka['Ключ'] = apteka['Номер накладной'].astype(str)
            mask_ea = apteka['Поставщик'].astype(str).str.contains('ЕАПТЕКА', na=False)
            apteka.loc[mask_ea, 'Ключ'] += '/15'
            
            # Соединяем со справочником СБИС (LEFT JOIN)
            apteka = apteka.merge(sbis_lookup, left_on='Ключ', right_on='Номер_поиска', how='left')
            
            # Заполняем номер счёта-фактуры (берём оригинальный номер из СБИС)
            apteka['Номер счет-фактуры'] = apteka['Ключ'].map(
                dict(zip(sbis_lookup['Номер_поиска'], sbis_grouped['Номер']))
            ).fillna('')
            
            # Заполняем сумму и дату
            apteka['Сумма счет-фактуры'] = apteka['Сумма_счета'].fillna(0)
            apteka['Дата счет-фактуры'] = apteka['Дата_счета'].fillna('')
            
            # Сравниваем даты
            date_sf = apteka['Дата счет-фактуры'].astype(str)
            date_nakl = apteka['Дата накладной'].astype(str)
            
            apteka['Сравнение дат'] = ''
            # Случай 1: счёт-фактура не найден, но дата накладной есть
            apteka.loc[(date_sf == '') & (date_nakl != '') & (date_nakl != 'nan'), 'Сравнение дат'] = 'Не совпадает!'
            # Случай 2: счёт-фактура найден, но даты разные
            apteka.loc[(date_sf != '') & (date_sf != date_nakl), 'Сравнение дат'] = 'Не совпадает!'
            
            # Удаляем временные колонки
            apteka = apteka.drop(columns=['Ключ', 'Номер_поиска', 'Сумма_счета', 'Дата_счета'], errors='ignore')
            
            # Оставляем только нужные колонки в правильном порядке
            apteka = apteka[[c for c in apteka_columns if c in apteka.columns]]
            
            # Сохраняем результат
            file_name = os.path.basename(file_path)
            output_path = f"{result_path_for_folder}{os.path.splitext(file_name)[0]} - результат.xlsx"
            apteka.to_excel(output_path, index=False)
            print(f'{file_name} -> {output_path}')

366.csv Обработан!
А123.csv Обработан!
